# Cyrus, Winnowed

In [41]:
from winnow_fcns import *
from sim_utils import *
import numpy as np
from pathlib import Path
import os

In [42]:
class MockBitBuffer:
    def __init__(self, bits):
        self.bits = list(bits)
        self.seed = None
    def get_length(self): return len(self.bits)
    def get_bit(self, i): return self.bits[i]
    def set_bit(self, i): self.bits[i] = 1
    def clear_bit(self, i): self.bits[i] = 0
    def flip_bit(self, i): self.bits[i] = 1 - self.bits[i]
    def set_seed(self, s): self.seed = s
    def permute_buffer(self): pass # Simplified for test

In [43]:
# rng = np.random.default_rng(seed=42)
# rng2 = np.random.default_rng(seed=180)
# N = 30
# std = 0.1
# ber = 0.25


# init_key = rng2.integers(low=0, high=2, size=N)


# print(init_key)
# key = simulate_error(init_key, rng, ber=ber, N=N)
# print(key)


#------------------files---------------------------------------------
current_file = Path.cwd()

init_file = current_file.parent / "initial_cyrus_bits.csv"
ch1_file = current_file.parent / "ch1_cyrus.csv"
ch2_file = current_file.parent / "ch2_cyrus.csv"



def load_data(init_file, ch1_file, ch2_file):
    init = np.loadtxt(init_file, delimiter=',', dtype=int)
    ch1 = np.loadtxt(ch1_file, delimiter=',')
    ch2 = np.loadtxt(ch2_file, delimiter=',')
    return init, ch1, ch2


init_key, ch1, ch2 = load_data(init_file, ch1_file, ch2_file)


In [44]:
alice_key = MockBitBuffer(list(ch1))
bob_key   = MockBitBuffer(list(ch2))


#  initialize Alice and Bob
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)
#use same seed so they dont shuffle keys differently!!!!

# start first pass
alice_winnow.first_pass()
bob_winnow.first_pass()

# alice calculates syndrome for the first block 
alice_syndrome = alice_winnow.get_syndrome(0)

# bob uses Alice's syndrome to fix his key
print(f"Bob's key before fix: {bob_key.bits[:8]}")
bob_winnow.fix_with_syndrome(0, alice_syndrome)
print(f"Bob's key after fix:  {bob_key.bits[:8]}")
print(f"Alice's key: {alice_key.bits[:8]}")

# verify that the keys match
if alice_key.bits == bob_key.bits:
    print(" Success! Bob corrected the error.")
else:
    print("Failure: Keys still do not match.")

TypeError: ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''